FAISS = Facebook Ai Similarity Search used to create vector database for similarity search

In [ ]:
import os

print(os.getcwd())
print(os.listdir("Group IV Syllabus.pdf"))

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# loader = PyPDFLoader("Ai_with_Rag\Group IV Syllabus.pdf")
loader = PyPDFLoader(
    r"D:\JEEVA\GENAI\Ai_with_Rag\Group IV Syllabus.pdf"
)
docs = loader.load()

splitter=RecursiveCharacterTextSplitter(
    chunk_size=800,chunk_overlap=100
)

chunks = splitter.split_documents(docs)
len(chunks)




47

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

db=FAISS.from_documents(chunks,embeddings)
retriever = db.as_retriever()



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2483.83it/s]


In [1]:
#error in this part

from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_token=256,
    temperature=0.2
)

llm=HuggingFacePipeline(pipeline=pipe)

print("Model loaded successfully!")



d:\JEEVA\GENAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Admin\AppData\Local\Temp\ipykernel_21436\1434023426.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline
d:\JEEVA\GENAI\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled 

KeyboardInterrupt: 

In [ ]:
def rag_chain(question):
    docs = retriever.invoke(question)
    
    context = "\n\n".join([d.page_content for d in docs])
    
    prompt_text = f"""
    you are an AI assistant, Use ONly the provided content to answer.
    If the asnwer is not in the context, say "I dont Know".
    
    Context:
    {context}
    Question:{question}
    
    Asnwer:
    """
    
    raw = llm.invoke(prompt_text)
    answer= raw.replace(prompt_text,"").strip
    return answer
    
    

In [ ]:
response = rag_chain("what that pdf contains")
print(response)